## Modules import

In [5]:
import pandas as pd
import json
import time
from clickhouse_driver import Client
from typing import Optional
import plotly.express as px

## Connection to ClickHouse

### Dataset info

Data being analyzed here is a part of old commercial advertisements logs, provided to me as a processable example for the task. It is not covered by NDA and has neglectable business value.

Dataset includes 2 tables: `E.TrackDays` and `E.Tracking3` of structure and settings described below.

#### Table 1 - E.TrackDays

##### Description

**Purpose:** Aggregated daily advertising statistics.

**Engine:** `SummingMergeTree`. 

##### Columns

**Dimensions:**

| Field           | Type     | Description |
|-----------------|----------|-------------|
| `dt`            | `Date`   | Event date |
| `demand_id`     | `UInt32` | Demand source ID |
| `demand_name`   | `String` | Demand source name |
| `router_id`     | `UInt32` | Request router ID |
| `router_uid`    | `String` | Router unique string ID |
| `router_name`   | `String` | Router name |
| `tag_id`        | `UInt32` | Placement (tag) ID |
| `tag_uid`       | `String` | Tag unique string ID |
| `tag_name`      | `String` | Tag name |
| `channel_id`    | `UInt32` | Sales channel / traffic source ID |
| `channel_uid`   | `String` | Channel unique string ID |
| `channel_name`  | `String` | Channel name |
| `campaign_id`   | `UInt32` | Campaign ID |
| `campaign_uid`  | `String` | Campaign unique string ID |
| `campaign_name` | `String` | Campaign name |
| `creative_id`   | `UInt32` | Creative ID |
| `creative_uid`  | `String` | Creative unique string ID |
| `creative_name` | `String` | Creative name |
| `adv_id`        | `UInt32` | Advertiser ID |
| `adv_name`      | `String` | Advertiser name |
| `sub_id`        | `UInt32` | Sub-advertiser / agency ID |
| `sub_name`      | `String` | Sub-advertiser name |
| `app_name`      | `String` | Application / site name |
| `app_bundle`    | `String` | Application bundle ID |
| `country`       | `String` | Country (geo) |
| `device`        | `String` | Device type |
| `os`            | `String` | Operating system |
| `size`          | `String` | Placement size |
| `reason`        | `String` | Reason / event type code |
| `device2`       | `String` | Additional device detail |
| `req_type`      | `Enum8(''=0,'VAST'=1,'RTB'=2)` | Request protocol |

**Prices:**

| Field | Type | Description |
|-------|------|-------------|
| `system_price` | `Decimal(18,10)` | System price (e.g. base rate) |
| `tag_price` | `Decimal(18,8)` | Price for the placement |
| `channel_price` | `Decimal(18,8)` | Price for the channel |

**Metrics:**

All metrics are `UInt64` and are summed automatically.

| Field | Description |
|-------|-------------|
| `reqs` | Ad requests |
| `req2` | Additional request counter (e.g. filtered) |
| `opps` | Opportunities to serve |
| `opp2` | Refined opportunities |
| `ress` | Responses |
| `res2` | Additional response counter |
| `imps` | Impressions |
| `invs` | In‑view / engagements |
| `opp3` | Extra opportunities |
| `opp3t` | Total extra opportunities |

**Key Settings**  
- `PARTITION BY toYYYYMM(dt)`  
- `PRIMARY KEY (demand_id, dt)`  
- `ORDER BY (demand_id, dt, demand_name, tag_id, tag_uid, tag_name, channel_id, channel_uid, channel_name, adv_id, adv_name, sub_id, sub_name, app_name, app_bundle, country, device, os, size, reason, device2, router_id, campaign_id, creative_id)`  
- `index_granularity = 8192` 

> **Important:** Always query with `GROUP BY` on the full sorting key and `SUM()` on metrics to obtain accurate results (parts may not be fully merged).

#### Table 2 - E.Tracking3

##### Description

**Purpose:** Aggregated real‑time advertising tracking statistics.  

**Engine:** `SummingMergeTree`

##### Columns

**Dimensions:**

| Field | Type | Description |
|-------|------|-------------|
| `dt` | `DateTime` | Event timestamp |
| `demand_id` | `UInt32` | Demand source ID |
| `demand_name` | `String` | Demand source name |
| `router_id` | `UInt32` | Request router ID |
| `router_uid` | `String` | Router unique string ID |
| `router_name` | `String` | Router name |
| `tag_id` | `UInt32` | Placement (tag) ID |
| `tag_uid` | `String` | Tag unique string ID |
| `tag_name` | `String` | Tag name |
| `adv_id` | `UInt32` | Advertiser ID |
| `adv_name` | `String` | Advertiser name |
| `channel_id` | `UInt32` | Sales channel / traffic source ID |
| `channel_uid` | `String` | Channel unique string ID |
| `channel_name` | `String` | Channel name |
| `campaign_id` | `UInt32` | Campaign ID |
| `campaign_uid` | `String` | Campaign unique string ID |
| `campaign_name` | `String` | Campaign name |
| `creative_id` | `UInt32` | Creative ID |
| `creative_uid` | `String` | Creative unique string ID |
| `creative_name` | `String` | Creative name |
| `sub_id` | `UInt32` | Sub-advertiser / agency ID |
| `sub_name` | `String` | Sub-advertiser name |
| `app_name` | `String` | Application name |
| `app_bundle` | `String` | Application bundle ID |
| `app_url` | `String` | Application URL |
| `site_name` | `String` | Website name |
| `site_domain` | `String` | Website domain |
| `site_url` | `String` | Website URL |
| `dooh_id` | `String` | Digital out‑of‑home identifier |
| `venue_type_id` | `UInt32` | Venue type ID (DOOH) |
| `country` | `String` | Country (geo) |
| `state` | `String` | State / region |
| `city` | `String` | City |
| `zip` | `String` | ZIP / postal code |
| `device` | `String` | Device type |
| `os` | `String` | Operating system |
| `size` | `String` | Placement size (e.g. 320x50) |
| `reqSize` | `String` | Requested size |
| `reason` | `String` | Reason / status code |
| `device2` | `String` | Additional device detail |
| `pubid` | `String` | Publisher ID |
| `pubid2` | `String` | Additional publisher ID |
| `iab` | `String` | IAB category |
| `crid` | `String` | Creative ID (external) |
| `contentCategory` | `String` | Content category |
| `contentTitle` | `String` | Content title |
| `contentNetwork` | `String` | Content network |
| `contentGenre` | `String` | Content genre |
| `contentLanguage` | `String` | Content language |
| `contentRating` | `String` | Content rating |
| `contentChannel` | `String` | Content channel |
| `coppa` | `UInt8` | COPPA flag (0 or 1) |
| `req_type` | `Enum8(''=0,'VAST'=1,'RTB'=2)` | Request protocol |
| `imp_type` | `String` | Impression type |

**Prices & Bids:**

| Field | Type | Description |
|-------|------|-------------|
| `system_price` | `Decimal(18,10)` | System price (base rate) |
| `tag_price` | `Decimal(18,8)` | Price for the placement |
| `tag_cpm` | `Decimal(9,4)` | Tag CPM |
| `channel_price` | `Decimal(18,8)` | Price for the channel |
| `channel_cpm` | `Decimal(9,4)` | Channel CPM |
| `bidFloor` | `Decimal(18,8)` | Bid floor price |
| `multiplier` | `Decimal(18,2)` | Multiplier applied |

**Metrics:**

All metrics are `UInt64` and are summed automatically.

| Field | Description |
|-------|-------------|
| `reqs` | Ad requests |
| `opps` | Opportunities to serve |
| `ress` | Responses |
| `imps` | Impressions |
| `req2` | Additional request counter (e.g. filtered) |
| `opp2` | Refined opportunities |
| `res2` | Additional response counter |
| `opp3` | Extra opportunities |
| `opp3s` | Extra opportunities (supply) |
| `opp3m` | Extra opportunities (mediation) |
| `opp3t` | Total extra opportunities |
| `req3` | Additional request counter |
| `wins` | Won bids |
| `invs` | In‑view / engagements |
| `event_0` | Event: start |
| `event_25` | Event: 25% progress |
| `event_50` | Event: 50% progress |
| `event_75` | Event: 75% progress |
| `event_100` | Event: 100% completion |
| `clicks` | Clicks |

**Key Settings:**  
- `PARTITION BY toYYYYMMDD(dt)` - daily partitions  
- `PRIMARY KEY demand_id`  
- `ORDER BY (demand_id, demand_name, tag_id, tag_uid, tag_name, tag_cpm, channel_id, channel_uid, channel_name, channel_cpm, adv_id, adv_name, sub_id, sub_name, app_name, app_bundle, country, device, os, size, reason, device2, dt, iab, pubid, crid, contentCategory, contentTitle, contentNetwork, contentGenre, coppa, pubid2, reqSize, imp_type, site_name, site_domain, state, city, zip, router_id, campaign_id, creative_id, dooh_id, venue_type_id, contentLanguage, contentRating, contentChannel)`  
- `storage_policy = 'tracking3'`  
- `index_granularity = 8192`  

> **Important:** Always query with `GROUP BY` on the full sorting key and `SUM()` on metrics to obtain accurate results (parts may not be fully merged).

### Connection to database

In [ ]:
with open("logging_config.json", "r") as logging_config_file:
    logging_config = json.load(logging_config_file)

In [ ]:
click_client = Client(
    host=logging_config["host"],
    port=logging_config["port"],
    user=logging_config["user"],
    password=logging_config["password"],
    database=logging_config["database"],
    connect_timeout=60,
    send_receive_timeout=600
)

## SPU count

In [3]:
delta_seconds = 1800
tracking3_name = "Tracking3"
trackdays_name = "TrackDays"

dates_spu_count = ("2024-01-01", "2024-02-01")

In [4]:
query_spu = f"""
SELECT avg(sessions_per_user) AS spu
FROM (
    SELECT
        concat(device, '_', os, '_', country, '_', city) AS user_id,
        sum(is_new_session) AS sessions_per_user
    FROM (
        SELECT device, os, country, city, dt,
            IF(
                lag(dt) OVER (PARTITION BY concat(device, '_', os, '_', country, '_', city) ORDER BY dt) IS NULL
                OR dateDiff('second', lag(dt) OVER (PARTITION BY concat(device, '_', os, '_', country, '_', city) ORDER BY dt), dt) > {delta_seconds},
                1, 0
            ) AS is_new_session
        FROM {tracking3_name}
        WHERE dt >= '{dates_spu_count[0]}' AND dt < '{dates_spu_count[1]}'
    )
    GROUP BY user_id
)
"""

spu_result = click_client.execute(query_spu)
print(f"SPU = {spu_result[0][0]:.2f}")

Failed to connect to 95.216.28.38:9000
Traceback (most recent call last):
  File "c:\Users\Элина\Desktop\Вузовские работы (магистратура)\1,2,3 CV,Осн.ML,NLP,Прак.ML\ML_homeworks_magistracy\.mlenv\Lib\site-packages\clickhouse_driver\connection.py", line 428, in connect
    return self._init_connection(host, port)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Элина\Desktop\Вузовские работы (магистратура)\1,2,3 CV,Осн.ML,NLP,Прак.ML\ML_homeworks_magistracy\.mlenv\Lib\site-packages\clickhouse_driver\connection.py", line 358, in _init_connection
    self.socket = self._create_socket(host, port)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Элина\Desktop\Вузовские работы (магистратура)\1,2,3 CV,Осн.ML,NLP,Прак.ML\ML_homeworks_magistracy\.mlenv\Lib\site-packages\clickhouse_driver\connection.py", line 326, in _create_socket
    raise err
  File "c:\Users\Элина\Desktop\Вузовские работы (магистратура)\1,2,3 CV,Осн.ML,NLP,Прак.ML\ML_homeworks_magistracy\.mlenv

SocketTimeoutError: Code: 209. Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера (95.216.28.38:9000)

## Graphing the sessions per user distribution

In [ ]:
query_sessions_distrn = f"""
WITH delta AS {delta_seconds}
SELECT 
    user_id,
    sum(is_new_session) AS sessions
FROM (
    SELECT
        concat(device, '_', os, '_', country, '_', city) AS user_id,
        dt,
        if(lag(dt) OVER (PARTITION BY user_id ORDER BY dt) IS NULL
           OR dateDiff('second', lag(dt) OVER (PARTITION BY user_id ORDER BY dt), dt) > delta,
           1, 0) AS is_new_session
    FROM Tracking3
    WHERE dt >= '2024-01-01' AND dt < '2024-02-01'
)
GROUP BY user_id
"""

result = click_client.execute(query_sessions_distrn)
df_sessions = pd.DataFrame(result, columns=['user_id', 'sessions'])

In [ ]:
fig = px.histogram(
    df_sessions, 
    x="sessions", 
    nbins=30, 
    title="Sessions per user distribution",
    labels={'sessions': 'Число сессий', 'count': 'Количество пользователей'}
)
fig.show()

## Aggregation time with & without sorting key

### Original table copying for experiment clarity

In [ ]:
tracking3_no_sort_name = "Tracking3_no_sort"
query_copy_no_sort = f"""
    CREATE TABLE {logging_config["database"]}.{tracking3_no_sort_name}
    ENGINE = SummingMergeTree
    PARTITION BY toYYYYMMDD(dt)
    ORDER BY dt
    AS SELECT * FROM {logging_config["database"]}.{tracking3_name}
"""

click_client.execute(query_copy_no_sort)

### Time measurement

In [ ]:
def measure_query_time(client: Client, 
                       query: str,
                       verbose: Optional[bool] = True):
    start = time.time()
    client.query(query)
    query_time = time.time() - start

    if verbose:
        print(f"Query took time: {query_time} ms")
    return query_time

In [ ]:
query_simple_aggregate = f"""
    SELECT demand_id, count() 
    FROM {logging_config["database"]}.{tracking3_no_sort_name} 
    GROUP BY demand_id
"""
time_no_sort = measure_query_time(click_client, query_simple_aggregate)
print(f"Aggregation time without sorting key: {time_no_sort:.2f} сек")

### Additional table drop

In [ ]:
click_client.command(f"DROP TABLE {logging_config["database"]}.{tracking3_no_sort_name}")